# Phase 1: Data Loading & Exploratory Data Analysis

**Movie Recommendation System — CiaoDVD Dataset**

This notebook covers Phase 1 of the project:
1. Load the raw ratings file
2. Report basic statistics (users, movies, ratings, sparsity)
3. Visualize the rating distribution
4. Visualize the long-tail (ratings per user / per movie)
5. Check data quality (missing, duplicates) and clean
6. Save the cleaned data to `data/processed/` for the modeling notebooks


In [ ]:
import sys
from pathlib import Path

# Allow `from src.xxx import yyy` when this notebook lives in /notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_ratings
from src.preprocessing import clean_ratings, basic_stats, print_stats

sns.set_theme(style="whitegrid")


## 1. Load the dataset

Download `movie-ratings.txt` from https://guoguibing.github.io/librec/datasets.html and place it in `data/raw/`.

In [ ]:
ratings_raw = load_ratings("data/raw/movie-ratings.txt")
print(f"Loaded {len(ratings_raw):,} ratings")
ratings_raw.head()


## 2. Basic statistics

In [ ]:
stats = basic_stats(ratings_raw)
print_stats(stats)


**Observations**

- The user–movie rating matrix is **>99.97% sparse** — only a tiny fraction of all
  possible (user, movie) pairs are observed.
- This is realistic for recommender systems and is the central challenge: most
  models have very little overlap to learn from.


## 3. Rating distribution

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(x="movieRating", data=ratings_raw)
plt.title("Distribution of Ratings")
plt.xlabel("Rating (1-5)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("results/figures/rating_distribution.png", dpi=150)
plt.show()


**Observations**

- Ratings are integers from 1 to 5.
- The distribution is **left-skewed** — most ratings are 4 or 5. Users tend to
  rate movies they liked.


## 4. Ratings per user and per movie (long-tail)

In [ ]:
ratings_per_user  = ratings_raw.groupby("userId").size()
ratings_per_movie = ratings_raw.groupby("movieId").size()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(ratings_per_user, bins=30, ax=axes[0])
axes[0].set_xlim(0, 100)
axes[0].set_xlabel("Number of Ratings per User")
axes[0].set_ylabel("Number of Users")
axes[0].set_title("Ratings per User")

sns.histplot(ratings_per_movie, bins=30, ax=axes[1])
axes[1].set_xlim(0, 100)
axes[1].set_xlabel("Number of Ratings per Movie")
axes[1].set_ylabel("Number of Movies")
axes[1].set_title("Ratings per Movie")

plt.tight_layout()
plt.savefig("results/figures/long_tail.png", dpi=150)
plt.show()


**Observations**

- Both distributions show a clear **long-tail pattern**.
- Most users rate only a few movies; a small group of "power users" account for
  many ratings. Same shape on the movie side.
- This long-tail behavior is what makes the rating matrix sparse and is a known
  challenge for collaborative filtering — especially KNN.


## 5. Data quality checks

In [ ]:
print("Missing values per column:")
print(ratings_raw.isnull().sum())
print()
print("Duplicate rows:", ratings_raw.duplicated().sum())
print("Unique rating values:", sorted(ratings_raw["movieRating"].unique()))


## 6. Clean and save

Drop unused columns (`reviewId`, `reviewDate`) and save to `data/processed/` for the modeling notebooks.

In [ ]:
ratings = clean_ratings(ratings_raw)
ratings.to_csv("data/processed/ratings_clean.csv", index=False)
print(f"Saved cleaned ratings to data/processed/ratings_clean.csv ({len(ratings):,} rows)")
ratings.head()


**Cleaning summary**

- No missing values, no imputation needed.
- No duplicate rows.
- All ratings are integers in [1, 5] — no invalid values.
- `reviewId` and `reviewDate` removed (not used by the recommendation models).
- `userId`, `movieId`, `movieRating`, and `movie_categoryId` are kept.

**Next step:** open `02_cf_model.ipynb` and `03_svd_model.ipynb` to train the models.